<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install dependencies
!pip install -q --upgrade "torchao>=0.16.0" "diffusers>=0.43.0" transformers accelerate peft safetensors pillow opencv-python-headless fastapi uvicorn python-multipart huggingface_hub requests

from IPython.display import clear_output
clear_output()
print("✅ Installed")

In [ ]:
#@title 2. Download & verify Qwen-Image 2.1 + BFS Head Swap LoRA weights
#@markdown **Head swap LoRA** (both by the BFS author, for Qwen-Image 2.1):
#@markdown - `v1.1`: recommended. Best skin detail and head angle.
#@markdown - `v1.1 alternative`: copies the expression better (open mouth, strong emotions); head angle is a little less exact.
HEAD_LORA = "v1.1 (recommended)" #@param ["v1.1 (recommended)", "v1.1 alternative (stronger expressions)"]
import gc
import os
import torch
from huggingface_hub import hf_hub_download, snapshot_download

torch.cuda.empty_cache()
gc.collect()

LORA_DIR = "/content/loras"
os.makedirs(LORA_DIR, exist_ok=True)

LORA_FILE = {
    "v1.1 (recommended)": "bfs_head_v1.1_qwen_2.1.safetensors",
    "v1.1 alternative (stronger expressions)": "bfs_head_v1.1_alternative_qwen_2.1.safetensors",
}[HEAD_LORA]
os.environ["HEADSWAP_BASE_LORA"] = LORA_FILE  # read by the server started in cell 4

print("Downloading/verifying base model (Qwen/Qwen-Image-2.1)...")
snapshot_download(
    repo_id="Qwen/Qwen-Image-2.1",
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
)

print(f"Downloading BFS Head Swap LoRA ({LORA_FILE})...")
lora_path = hf_hub_download(
    repo_id="Alissonerdx/BFS-Best-Face-Swap",
    filename=LORA_FILE,
    local_dir=LORA_DIR,
)

with open(lora_path, "rb") as f:
    header = f.read(16)
if header.startswith(b"{") or header.startswith(b"<"):
    raise RuntimeError(f"Downloaded file is not a valid safetensors binary: {lora_path}")

lora_size_mb = os.path.getsize(lora_path) / (1024 * 1024)
print(f"✅ Weights cached. LoRA path: {lora_path} ({lora_size_mb:.2f} MB)")
print("ℹ️ Don't add a body swap LoRA (bfs_body_swap_...) for head swaps: it is made for a different job and lowers quality.")

In [ ]:
#@title 3A. Clear kernel VRAM
import gc
import torch

if "pipe" in globals():
    del pipe

gc.collect()
torch.cuda.empty_cache()
print("✅ Notebook kernel GPU memory cleared.")

In [ ]:
%%writefile /content/headswap_server.py
# Head Swap backend: Qwen-Image 2.1 + BFS Head Swap LoRA, served for the website's Head Swap tab.
import os

# Must be set before torch is imported: lets PyTorch reuse fragmented GPU memory between requests.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import base64
import gc
import glob
import math
import random
import re
import threading
import time
import traceback
import urllib.parse
import uuid
from collections import deque
from contextlib import asynccontextmanager
from io import BytesIO
from typing import Optional

import requests
import torch
from diffusers import DiffusionPipeline
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from pydantic import BaseModel
from safetensors.torch import load_file

pipe: Optional[DiffusionPipeline] = None
LOG_BUFFER = deque(maxlen=200)
LORA_DIR = "/content/loras"
# The head swap LoRA, always on. Chosen in cell 2 (v1.1 = recommended, alternative = stronger expressions).
BASE_LORA = os.environ.get("HEADSWAP_BASE_LORA", "bfs_head_v1.1_qwen_2.1.safetensors")
BASE_ADAPTER = "bfs_head"
MODEL_EXTS = (".safetensors",)
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")
PIPE_LOCK = threading.Lock()   # one swap at a time on the GPU
LOADED_ADAPTERS = {}           # LoRA file name -> adapter name
JOBS = {}

# Quality settings, matching the official Qwen-Image 2.1 pipeline (BFS guide: LoRA strength 1.0, cfg 1).
DEFAULT_STEPS = 40          # Qwen-Image 2.1 is made for 40-50 steps without a speed LoRA; 20 leaves results soft
DEFAULT_RESOLUTION = 1024   # output and reference images are ~1024x1024 pixels in area (the model's default)
MAX_RESOLUTION = 1536       # higher looks sharper but needs more memory; falls back automatically on the L4
MULTIPLE = 32               # Qwen-Image 2.1 sizes must be multiples of 32

DEFAULT_PROMPT = (
    "head_swap: start with <image1> as the base image, keeping its lighting, environment, and background. "
    "remove the head from <image1> completely and replace it with the head from <image2>, strictly preserving "
    "the hair, eye color, nose structure from <image2>. copy the direction of the eye, head rotation, micro "
    "expressions from <image1>, high quality, sharp details, 4k"
)


def log_msg(msg: str) -> None:
    line = str(msg)
    LOG_BUFFER.append(line)
    print(line, flush=True)


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def find_lora_file(lora_dir: str = LORA_DIR) -> str:
    # Always the HEAD swap LoRA. (Picking the first file alphabetically would load bfs_body_swap_... instead.)
    preferred = os.path.join(lora_dir, BASE_LORA)
    if os.path.exists(preferred):
        return preferred
    candidates = sorted(glob.glob(os.path.join(lora_dir, "bfs_head*.safetensors")))
    if not candidates:
        raise FileNotFoundError(f"The BFS head swap LoRA ({BASE_LORA}) is missing from {lora_dir}. Run cell 2 again.")
    return candidates[0]


def list_lora_files():
    return sorted(f for f in os.listdir(LORA_DIR) if f.lower().endswith(MODEL_EXTS)) if os.path.isdir(LORA_DIR) else []


def adapter_name_for(file_name: str) -> str:
    return "user_" + re.sub(r"[^0-9a-zA-Z_]", "_", os.path.splitext(file_name)[0])[:60]


def load_lora(pipeline, path: str, adapter_name: str) -> None:
    try:
        pipeline.load_lora_weights(path, adapter_name=adapter_name)
    except Exception as lora_err:
        log_msg(f"Standard load_lora_weights failed for {os.path.basename(path)} ({lora_err}); retrying with remapped keys")
        try:
            pipeline.load_lora_weights(map_lora_state_dict(load_file(path)), adapter_name=adapter_name)
        except Exception as err:
            raise ValueError(
                f"LoRA '{os.path.basename(path)}' could not be loaded. It is probably not a Qwen-Image 2.1 LoRA. ({err})"
            ) from err


def map_lora_state_dict(state_dict: dict) -> dict:
    """Normalizes ComfyUI/Civitai LoRA prefixes while preserving Qwen-Image block names."""
    mapped = {}
    for k, v in state_dict.items():
        nk = k
        if nk.startswith("diffusion_model."):
            nk = nk[len("diffusion_model."):]
        if not nk.startswith("transformer.") and "transformer_blocks." in nk:
            nk = "transformer." + nk
        mapped[nk] = v
    return mapped


def load_pipeline() -> DiffusionPipeline:
    free_gpu()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    log_msg(f"Loading Qwen-Image 2.1 in {dtype} ...")

    pipeline = DiffusionPipeline.from_pretrained("Qwen/Qwen-Image-2.1", torch_dtype=dtype)

    lora_file = find_lora_file(LORA_DIR)
    log_msg(f"Loading head swap LoRA: {lora_file}")
    load_lora(pipeline, lora_file, BASE_ADAPTER)
    LOADED_ADAPTERS.clear()
    LOADED_ADAPTERS[os.path.basename(lora_file)] = BASE_ADAPTER

    # The model is ~32 GB in bf16 (transformer ~14 GB + text encoder ~17.5 GB), more than an L4 (22 GB) holds.
    # Only put everything on the GPU when it clearly fits (A100 80 GB, H100); otherwise keep one part on the GPU
    # at a time. Never try a full .to("cuda") first: a failed move leaves memory behind and later swaps run out.
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if total_gb >= 45:
        pipeline.to("cuda")
        log_msg(f"GPU has {total_gb:.0f} GB: whole model on the GPU.")
    else:
        pipeline.enable_model_cpu_offload()
        log_msg(f"GPU has {total_gb:.0f} GB: model CPU offload (one part on the GPU at a time).")

    vae = getattr(pipeline, "vae", None)
    if vae is not None:
        if hasattr(vae, "enable_tiling"):
            vae.enable_tiling()
        if hasattr(vae, "enable_slicing"):
            vae.enable_slicing()

    free_gpu()
    return pipeline


@asynccontextmanager
async def lifespan(app: FastAPI):
    global pipe
    pipe = load_pipeline()
    log_msg("Model + LoRA loaded and ready")
    yield
    pipe = None
    free_gpu()


app = FastAPI(lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class LoraChoice(BaseModel):
    name: str
    strength: float = 1.0


class HeadSwapRequest(BaseModel):
    body: str
    head: str
    prompt: str = ""
    num_inference_steps: Optional[int] = None
    resolution: Optional[int] = None
    seed: Optional[int] = None
    loras: list[LoraChoice] = []   # extra LoRAs on top of the head swap LoRA


class ModelDownload(BaseModel):
    url: str
    type: str = "lora"


def dataurl_to_pil(dataurl: str) -> Image.Image:
    if "," in dataurl:
        dataurl = dataurl.split(",", 1)[1]
    return Image.open(BytesIO(base64.b64decode(dataurl))).convert("RGB")


def pil_to_dataurl(img: Image.Image) -> str:
    buffered = BytesIO()
    img.convert("RGB").save(buffered, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffered.getvalue()).decode("utf-8")


def output_size(img: Image.Image, resolution: int) -> tuple:
    """Body image's aspect ratio at about resolution x resolution pixels, in multiples of 32."""
    w, h = img.size
    ratio = w / h
    out_w = math.sqrt(resolution * resolution * ratio)
    out_h = out_w / ratio
    out_w = max(MULTIPLE, int(round(out_w / MULTIPLE)) * MULTIPLE)
    out_h = max(MULTIPLE, int(round(out_h / MULTIPLE)) * MULTIPLE)
    return out_w, out_h


def new_job(kind, **extra):
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"kind": kind, "status": "queued", "created": time.time(), "error": None, **extra}
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    return job_id


@app.get("/health")
def health():
    return {"status": "ok", "loaded": pipe is not None}


@app.get("/logs")
def get_logs():
    return {"logs": list(LOG_BUFFER), "lines": list(LOG_BUFFER)}


@app.get("/user_models")
def user_models():
    return {
        "model_type": "qwen_image_21_headswap",
        # Other BFS head swap files are left out: two head swap LoRAs at once fight each other
        "lora_files": [f for f in list_lora_files() if not f.startswith("bfs_head")],
        "base_lora": BASE_LORA,
        "supports_loras": True,
    }


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": "Unknown job (the server restarted). Try again."}
    return {**job, "job_id": job_id}


# ---------------- LoRA downloads (Civitai / Hugging Face links) ----------------
def resolve_url(url):
    """Turn page links into direct download links. Returns (download_url, suggested_name)."""
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    token = (qs.get("token") or [CIVITAI_TOKEN])[0]
    host = parsed.netloc.lower()

    if "civitai.com" in host or "civitai.red" in host:
        api_headers = {"Authorization": f"Bearer {token}"} if token else {}
        m = re.search(r"/api/download/models/(\d+)", parsed.path)
        if m:
            version_id = m.group(1)
        else:
            version_id = (qs.get("modelVersionId") or [None])[0]
            if not version_id:
                mm = re.search(r"/models/(\d+)", parsed.path)
                if not mm:
                    raise ValueError("Unrecognised Civitai link. Use a model page or download link.")
                info = requests.get(f"https://civitai.com/api/v1/models/{mm.group(1)}", headers=api_headers, timeout=30)
                info.raise_for_status()
                versions = info.json().get("modelVersions") or []
                if not versions:
                    raise ValueError("That Civitai model has no downloadable versions.")
                version_id = versions[0]["id"]  # newest version
        url = f"https://civitai.com/api/download/models/{version_id}"
        if token:
            url += "?token=" + urllib.parse.quote(token)
        return url, None

    if "huggingface.co" in host:
        url = url.replace("/blob/", "/resolve/")
        return url, os.path.basename(urllib.parse.urlparse(url).path)

    return url, os.path.basename(parsed.path)


def filename_from_response(resp, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
    name = urllib.parse.unquote(m.group(1).strip()) if m else (fallback or "")
    name = os.path.basename(name).strip().strip('"')
    return re.sub(r"[^\w.\-+() ]", "_", name)


def run_download(job_id, url):
    job = JOBS[job_id]
    job["status"] = "running"
    tmp_path = None
    try:
        os.makedirs(LORA_DIR, exist_ok=True)
        dl_url, suggested = resolve_url(url)
        with requests.get(dl_url, stream=True, timeout=60, allow_redirects=True,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            if r.status_code in (401, 403):
                raise ValueError("Access denied: this model needs a Civitai API key")
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError("The link returned a web page, not a model file. Use the download link.")
            name = filename_from_response(r, suggested)
            if not name.lower().endswith(MODEL_EXTS):
                name = f"lora_{int(time.time())}.safetensors"
            job["filename"] = name
            path = os.path.join(LORA_DIR, name)
            if os.path.exists(path):
                job["status"], job["progress"] = "done", 100
                return
            total = int(r.headers.get("content-length") or 0)
            job["total_mb"] = round(total / 1e6) if total else None
            tmp_path = path + ".part"
            done = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
                    done += len(chunk)
                    job["done_mb"] = round(done / 1e6)
                    job["progress"] = round(done * 100 / total) if total else None
            os.replace(tmp_path, path)
            tmp_path = None
        job["status"], job["progress"] = "done", 100
        log_msg(f"Downloaded LoRA: {name} ({job.get('done_mb')} MB)")
    except Exception as e:
        log_msg(traceback.format_exc())
        job["status"], job["error"] = "error", str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


@app.post("/download_model")
def download_model(req: ModelDownload):
    job_id = new_job("download", type="lora", filename=None, progress=0, done_mb=0, total_mb=None)
    threading.Thread(target=run_download, args=(job_id, req.url), daemon=True).start()
    return {"job_id": job_id, "status": "queued", "type": "lora"}


def apply_loras(choices) -> str:
    """Turn on the head swap LoRA plus the chosen extra LoRAs; all other loaded LoRAs are switched off."""
    names, weights = [BASE_ADAPTER], [1.0]
    for choice in choices:
        file_name = os.path.basename(choice.name)
        if file_name == BASE_LORA:
            weights[0] = float(choice.strength)
            continue
        path = os.path.join(LORA_DIR, file_name)
        if not os.path.exists(path):
            raise ValueError(f"LoRA '{file_name}' is not on the server. Download it again.")
        if file_name not in LOADED_ADAPTERS:
            log_msg(f"Loading LoRA {file_name} ...")
            load_lora(pipe, path, adapter_name_for(file_name))
            LOADED_ADAPTERS[file_name] = adapter_name_for(file_name)
        names.append(LOADED_ADAPTERS[file_name])
        weights.append(float(choice.strength))
    pipe.set_adapters(names, adapter_weights=weights)
    return ", ".join(f"{n}@{w:g}" for n, w in zip(names, weights))


def run_pipe(job, body_img, head_img, prompt, steps, resolution, use_kv_cache, seed):
    width, height = output_size(body_img, resolution)
    log_msg(f"Running head swap: {width}x{height}, steps={steps}, kv_cache={use_kv_cache}, seed={seed}")
    job.update(total_steps=steps, step=0, message=f"Generating {width}x{height}")

    def on_step_end(_pipe, step_index, _timestep, callback_kwargs):
        job.update(step=step_index + 1, progress=round(5 + 90 * (step_index + 1) / steps))
        log_msg(f"Step {step_index + 1}/{steps}")
        return callback_kwargs

    with torch.inference_mode():
        return pipe(
            prompt=prompt,
            image=[body_img, head_img],   # <image1> = body (kept), <image2> = head (pasted in) - keep this order
            width=width,                 # always pass the size: otherwise the pipeline uses the LAST image (the head)
            height=height,
            output_resolution=resolution,  # the pipeline also resizes both reference images to this area
            num_inference_steps=steps,
            true_cfg_scale=1.0,
            use_kv_cache=use_kv_cache,
            generator=torch.Generator("cpu").manual_seed(seed),
            callback_on_step_end=on_step_end,
        ).images[0]


def run_headswap(job_id, req: HeadSwapRequest):
    job = JOBS[job_id]
    job.update(status="queued", message="Waiting for the GPU (another swap is running)")
    with PIPE_LOCK:
        job.update(status="running", message="Preparing images", progress=3)
        try:
            body_img = dataurl_to_pil(req.body)
            head_img = dataurl_to_pil(req.head)
            prompt = req.prompt.strip() or DEFAULT_PROMPT
            steps = max(8, min(60, int(req.num_inference_steps or DEFAULT_STEPS)))
            resolution = max(512, min(MAX_RESOLUTION, int(req.resolution or DEFAULT_RESOLUTION)))
            seed = int(req.seed) if req.seed is not None else random.randint(0, 2**31 - 1)

            job.update(message="Loading LoRAs")
            log_msg(f"LoRAs: {apply_loras(req.loras)}")

            # Try the fast setting first, then lighter settings if the GPU runs out of memory.
            # The KV cache speeds up each step but holds ~3 GB (more at higher resolution).
            attempts = [(resolution, True), (resolution, False)]
            if resolution > DEFAULT_RESOLUTION:
                attempts.append((DEFAULT_RESOLUTION, False))
            if resolution > 768:
                attempts.append((768, False))

            result, last_error = None, None
            for res, kv in attempts:
                free_gpu()
                try:
                    result = run_pipe(job, body_img, head_img, prompt, steps, res, kv, seed)
                    break
                except torch.OutOfMemoryError as oom:
                    last_error = oom
                    log_msg(f"Out of GPU memory at {res}px (kv_cache={kv}); trying a lighter setting...")
                    job.update(message="Out of GPU memory, retrying with a lighter setting")
            if result is None:
                raise RuntimeError(
                    "Out of GPU memory even at the lightest setting. Restart the Colab runtime and run all cells "
                    f"again, or use a bigger GPU (A100). Details: {last_error}"
                )

            # Back to the body photo's original size, so the result lines up with it
            if result.size != body_img.size:
                result = result.resize(body_img.size, Image.Resampling.LANCZOS)

            job.update(status="done", progress=100, message="Done", image=pil_to_dataurl(result), seed=seed)
            log_msg("Done.")
        except Exception as e:
            log_msg(traceback.format_exc())
            job.update(status="error", error=str(e) or type(e).__name__)
        finally:
            free_gpu()


@app.post("/headswap")
def headswap(req: HeadSwapRequest):
    if pipe is None:
        return {"error": "Pipeline not initialized"}
    # Runs in the background: a swap can take longer than the Cloudflare tunnel's ~100 s request limit.
    # The website polls /job/<id> for progress and the result.
    job_id = new_job("headswap", progress=0, step=0, total_steps=None, message="Queued", image=None)
    threading.Thread(target=run_headswap, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued"}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

In [ ]:
#@title 4. Start server + Cloudflare tunnel
import os
import re
import subprocess
import sys
import time
import urllib.request

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"],
    check=True,
)

SERVER_SCRIPT = "/content/headswap_server.py"
SERVER_LOG = "/content/headswap_server.log"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

subprocess.run(["pkill", "-f", "headswap_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            CLOUDFLARED,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

print("Starting Head Swap server...")
server_env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        [sys.executable, SERVER_SCRIPT],
        env=server_env,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Waiting for model to load and Uvicorn to bind port 8000...")
model_ready = False
for _ in range(450):  # loading the ~32 GB model can take several minutes
    time.sleep(2)
    if server_proc.poll() is not None:
        print("❌ Server process exited early. Check log below:")
        break
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
        if "Application startup complete" in content:
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as resp:
                    if resp.status == 200 and b'"loaded":true' in resp.read().replace(b" ", b""):
                        print("✅ Model loaded and /health is ready")
                        model_ready = True
                        break
            except Exception:
                pass

if not model_ready:
    print("⚠️ Model did not finish loading in time. Last 30 log lines:")
    with open(SERVER_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("HeadSwap server failed to become ready.")

print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
        break

if TUNNEL_URL:
    print("\n" + "=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("=" * 60)
    print("\n→ Copy the URL above and paste it into the Head Swap tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")

In [ ]:
!tail -50 /content/headswap_server.log